# Milestone 3 lab, notebook 3 of 8: Run it twice

## Where we are

| Notebook | What you do |
|---|---|
| 1. One order, one row | Write the same rows three ways: append, overwrite, upsert |
| 2. Which column is the key? | Merge on the wrong key. Then meet an order that arrives twice in one run |
| **3. Run it twice** (you are here) | Prove that a second run changes nothing |
| 4. At least once, exactly once | Crash the load in three places, and see what each one costs |
| 5. The update that came late | Find a change the watermark has passed. Catch it with a lookback |
| 6. Fix one day | Break Friday's numbers, and rebuild Friday alone |
| 7. The order that vanished | A deleted order, and why the query cannot see it |
| 8. The production version | The three properties, as tests |

At Arden, someone once ran the load two times on one morning, and the report doubled. Jobs run twice all the time: a person clicks again, a scheduler retries, someone re-runs to fix a bug. A pipeline must be built for that.

This notebook uses the finished pipeline of this milestone, and tries to make it double.

In [1]:
import pandas as pd

from lab_helpers import lab_warehouse, money, run_pipeline, show_doc, table

## Step 1: A switch and a toggle

Think of two kinds of light switch.

- A **toggle** changes the light each time you press it. Press it twice, and you are back where you started. You must know how often it was pressed.
- A switch marked **ON** sets the light on. Press it once or ten times: the light is on.

An append is a toggle. Each run adds the rows again. An upsert is the ON switch. It says what the row **should be**, and saying it twice changes nothing.

A job that behaves like the ON switch has a name: it is **idempotent**.

## Step 2: Monday, Tuesday, and Tuesday again

In [2]:
con = lab_warehouse(fresh=True)

def state():
    """What the warehouse holds now."""
    return table(con, """
        SELECT (SELECT count(*) FROM raw.orders_landing)                               AS raw_rows,
               (SELECT count(*) FROM core.orders)                                      AS orders,
               (SELECT sum(orders) FROM analytics.daily_orders WHERE order_date = '2026-06-05')        AS friday_orders,
               (SELECT sum(revenue_cents) FROM analytics.daily_orders WHERE order_date = '2026-06-05') AS friday_cents
    """).iloc[0].to_dict()

history = []
for label, day in [("Monday", "2026-06-08"), ("Tuesday", "2026-06-09"), ("Tuesday again", "2026-06-09"),
                   ("Tuesday, 3rd time", "2026-06-09"), ("Tuesday, 4th time", "2026-06-09")]:
    result = run_pipeline(con, day)
    history.append({"run": label, "run_id": result.run_id, "rows read": result.rows_landed,
                    "orders inserted": result.orders_inserted, "orders updated": result.orders_updated, **state()})

pd.DataFrame(history)

,run,run_id,rows read,orders inserted,orders updated,raw_rows,orders,friday_orders,friday_cents
0,Monday,2026-06-08_1,226,226,0,226,226,37,97682
1,Tuesday,2026-06-09_1,102,25,4,328,251,37,97104
2,Tuesday again,2026-06-09_2,62,0,0,390,251,37,97104
3,"Tuesday, 3rd time",2026-06-09_3,62,0,0,452,251,37,97104
4,"Tuesday, 4th time",2026-06-09_4,62,0,0,514,251,37,97104


Read the columns one by one.

- **rows read**: every Tuesday run read rows. The source was asked each time.
- **raw_rows**: the raw table grew each time. It is append-only, and that is fine: nobody counts from it.
- **orders inserted / updated**: after the first Tuesday run, zero and zero.
- **orders**, **friday_orders**, **friday_cents**: the same after one Tuesday run as after four.

## Step 3: Prove it, do not claim it

"It looks the same" is not proof. Compare every row of the clean table and the report.

In [3]:
def snapshot():
    orders = con.execute("SELECT order_id, status, total_cents, refunded_cents, updated_at, is_deleted FROM core.orders ORDER BY 1").fetchall()
    report = con.execute("SELECT * FROM analytics.daily_orders ORDER BY 1, 2").fetchall()
    return orders, report

before = snapshot()
run_pipeline(con, "2026-06-09")
after = snapshot()

print("Orders table identical after another run: ", before[0] == after[0], f"({len(after[0])} rows compared)")
print("Report table identical after another run: ", before[1] == after[1], f"({len(after[1])} rows compared)")

Orders table identical after another run:  True (251 rows compared)
Report table identical after another run:  True (24 rows compared)


That comparison is exactly what the test `test_running_the_same_morning_twice_changes_nothing` does.

## Step 4: Why the second run changed nothing

The merge has a third case that notebook 1 did not need.

In [4]:
show_doc("LLD.md", "4. The MERGE")

**From `docs/LLD.md`**

> ## 4. The MERGE
> 
> ```sql
> MERGE INTO core.orders AS t
> USING stage_orders AS s
>    ON t.order_id = s.order_id
> WHEN MATCHED AND s.updated_at > t.updated_at THEN      -- a newer version: update the row
>     UPDATE SET status = s.status, item_count = s.item_count,
>                total_cents = s.total_cents, refunded_cents = s.refunded_cents,
>                updated_at = s.updated_at, last_run_id = :run_id
> WHEN NOT MATCHED THEN                                   -- a new order: insert a row
>     INSERT (...) VALUES (...);
> ```
> 
> Three things can happen to an incoming row:
> 
> | The incoming order | Rule that fires | Result |
> |---|---|---|
> | Is not in the table | `NOT MATCHED` | Inserted |
> | Is in the table, and this version is newer | `MATCHED AND newer` | Updated |
> | Is in the table, with the same or an older version | none | **Left alone** |
> 
> The third row is what makes a second run harmless.
> 
> Order 1130 across two mornings:
> 
> | After run | Rows in `raw.orders_landing` | Row in `core.orders` |
> |---|---|---|
> | `2026-06-08_1` | 1 (refunded 0) | total 1475, refunded 0 |
> | `2026-06-09_1` | 2 (refunded 0, refunded 578) | total 1475, refunded **578** |
> | `2026-06-09_2` | 3 | total 1475, refunded 578: unchanged |

`WHEN MATCHED AND s.updated_at > t.updated_at`: an incoming row updates the table only when it is a **newer version**. On a second run every incoming row is the version the table already holds. No rule fires, and the row is left alone.

## Step 5: Compare with milestone 2

The same five runs, with milestone 2's append and with this milestone's merge:

| After | Milestone 2: rows the report counts | Milestone 3: rows the report counts |
|---|---|---|
| Monday | 226 | 226 |
| Tuesday | 255 | 251 |
| Tuesday again | 256 | 251 |
| Tuesday, 3rd time | 257 | 251 |
| Tuesday, 4th time | 258 | 251 |

The left column is from milestone 2's lab. The right column is the `orders` column above.

In [5]:
print("Friday 5 June after five runs:", int(history[-1]["friday_orders"]), "orders,", money(int(history[-1]["friday_cents"])))
con.close()

Friday 5 June after five runs: 37 orders, $971.04


## Two new words

| Word | Meaning | Here |
|---|---|---|
| **Idempotent** | "Idempotent means two runs leave the same result as one run." | The merge, and the day-by-day report |
| **Safe re-run** | A run you can repeat without fear. It is what an idempotent pipeline gives you | Tuesday, four times |

The definition is from the D8Loop chapter [Idempotency, Retries & Exactly-Once](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/idempotency-retries-and-exactly-once).

One rule keeps a job idempotent: **never let the answer depend on when the job ran.** This pipeline stores `loaded_at` in the raw table, but no number in the report uses it.

## What you now know

- An idempotent job gives the same result after one run or five.
- The raw table may grow on a re-run. The tables people read may not change.
- The merge updates only when the incoming version is newer, so a repeat matches no rule.
- You prove idempotency by comparing the tables after one run and after two.

**Say it in an interview:** "Retries are normal, so every write in my pipeline is idempotent. I have a test that runs the same morning twice and then five times, and compares every row of the orders table and the report with the result of one run. The interviewer's question 'what if the job runs twice?' has a one-word answer: nothing."

Next: [notebook 4](04_at_least_once_exactly_once.ipynb) looks at the promise underneath: at least once, or exactly once.